In [1]:
!pip install transformers datasets evaluate accelerate


   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
   ---------------------------------------- 12.0/12.0 MB 71.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/564.3 kB ? eta -:--:--
   --------------------------------------- 564.3/564.3 kB 22.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 18.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/26.1 MB ? eta -:--:--
   ------------------------------------ --- 23.6/26.1 MB 118.7 MB/s eta 0:00:01
   ---------------------------------------- 26.1/26.1 MB 96.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/241.3 MB ? eta -:--:--
   --- ----------------------------------- 24.4/241.3 MB 118.9 MB/s eta 0:00:02
   ------- ------------------------------- 48.0/241.3 MB 118.7 MB/s eta 0:00:02
   ----------- --------------------------- 73.1/241.3 MB 118.6 MB/s eta 0:00:02
   -----------

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
df = pd.read_csv('predictedemotionall.csv')
df.columns

Index(['Unnamed: 0.14', 'Unnamed: 0.13', 'Unnamed: 0.12', 'Unnamed: 0.11',
       'Unnamed: 0.10', 'Unnamed: 0.9', 'Unnamed: 0.8', 'Unnamed: 0.7',
       'Unnamed: 0.6', 'Unnamed: 0.5', 'Unnamed: 0.4', 'Unnamed: 0.3',
       'Unnamed: 0.2', 'Unnamed: 0', 'Unnamed: 0.1', 'msg_type', 'datetime',
       'title', 'thread_id', 'comment_id', 'topic_name_top', 'topic_name_leaf',
       'thread_text', 'date', 'body', 'emotion', 'llm_emotion', 'normllm_emo',
       'government_sentiment', 'health_sentiment', 'community_sentiment',
       'misinformation_sentiment', 'overall_sentiment', 'title_translated',
       'overall_sentiment_label', 'government_sentiment_label',
       'health_sentiment_label', 'community_sentiment_label',
       'misinformation_sentiment_label', 'processed_thread_text',
       'transprocessed_thread_text', 'fin_sentiment_score',
       'eng_sentiment_score', 'fin_sentiment_label', 'eng_sentiment_label',
       'topic_name', 'llmtopic_name', 'fisentiment', 'ensentiment',


In [3]:
df['predicted_emotion'].value_counts()

predicted_emotion
anger                     2828
neutral                   1686
disgust                    492
surprise                   407
sadness                    265
fear                       138
Skipped - Invalid Text     121
joy                         20
confusion                    1
sad                          1
Name: count, dtype: int64

In [4]:
import pandas as pd

# Load your dataset (replace with your actual file path)
#df = pd.read_csv("/content/predictedemotion.csv")

# List of target emotions
target_emotions = ["neutral", "anger", "disgust", "fear", "sadness", "surprise", "joy"]

# Filter rows where predicted_emotion is in the target list
filtered_df = df[df['predicted_emotion'].isin(target_emotions)]

# Save to a new CSV (optional)
filtered_df.to_csv("filteredemotion_datasetall.csv", index=False)

In [5]:
filtered_df['predicted_emotion'].value_counts()

predicted_emotion
anger       2828
neutral     1686
disgust      492
surprise     407
sadness      265
fear         138
joy           20
Name: count, dtype: int64

In [6]:
from datasets import load_dataset

dataset = load_dataset('csv', data_files='filteredemotion_datasetall.csv')  # columns: text, emotion
dataset = dataset['train'].train_test_split(test_size=0.2)


Generating train split: 0 examples [00:00, ? examples/s]

In [7]:
label_names = list(set(dataset['train']['predicted_emotion']))
label2id = {label: i for i, label in enumerate(label_names)}
id2label = {i: label for label, i in label2id.items()}

def encode_labels(example):
    example['label'] = label2id[example['predicted_emotion']]
    return example

dataset = dataset.map(encode_labels)


Map:   0%|          | 0/4668 [00:00<?, ? examples/s]

Map:   0%|          | 0/1168 [00:00<?, ? examples/s]

In [8]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "michellejieli/emotion_text_classifier"  # or any transformer model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)


tokenizer_config.json:   0%|          | 0.00/413 [00:00<?, ?B/s]

C:\Users\kvishal22\AppData\Local\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kvishal22\.cache\huggingface\hub\models--michellejieli--emotion_text_classifier. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [11]:
def tokenize_function(example):
    return tokenizer(example["body"], padding="max_length", truncation=True)

tokenized_dataset = dataset.map(tokenize_function, batched=True)


Map:   0%|          | 0/4668 [00:00<?, ? examples/s]

Map:   0%|          | 0/1168 [00:00<?, ? examples/s]

In [12]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./emotion-model",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
)


In [13]:
from transformers import Trainer, TrainingArguments
import evaluate

accuracy = evaluate.load("accuracy")

def compute_metrics(p):
    predictions, labels = p
    preds = predictions.argmax(-1)
    return accuracy.compute(predictions=preds, references=labels)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)


C:\Users\kvishal22\AppData\Local\Temp\ipykernel_53036\1845071871.py:11: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [14]:
trainer.train()


C:\Users\kvishal22\AppData\Local\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.868750,0.690925
2,0.857000,0.865834,0.705479
3,0.857000,0.960513,0.708048


C:\Users\kvishal22\AppData\Local\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\kvishal22\AppData\Local\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=876, training_loss=0.6695579807507938, metrics={'train_runtime': 6895.6645, 'train_samples_per_second': 2.031, 'train_steps_per_second': 0.127, 'total_flos': 1855238863785984.0, 'train_loss': 0.6695579807507938, 'epoch': 3.0})

In [15]:
trainer.save_model("./emotion-model")
tokenizer.save_pretrained("./emotion-model")


('./emotion-model\\tokenizer_config.json',
 './emotion-model\\special_tokens_map.json',
 './emotion-model\\vocab.json',
 './emotion-model\\merges.txt',
 './emotion-model\\added_tokens.json',
 './emotion-model\\tokenizer.json')

In [ ]:
#

In [17]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch

# --------------------------
# 1. Load Locally Saved Fine-Tuned Model
# --------------------------
model_dir = "./emotion-model"

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Create pipeline for prediction
emotion_classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

# --------------------------
# 2. Load Unseen Text Data
# --------------------------
# Replace with your file path
df = pd.read_csv("nokianer.csv")  # Should have a column named "text"

# Drop rows with missing text
df = df.dropna(subset=['thread_text_translated'])

# --------------------------
# 3. Predict Emotions
# --------------------------
predictions = emotion_classifier(df['thread_text_translated'].tolist(), truncation=True)

# Add predicted labels to dataframe
df['predictedemotion'] = [pred['label'] for pred in predictions]

# --------------------------
# 4. Save Results
# --------------------------
df.to_csv("nokia_predicted_output.csv", index=False)
print("✅ Predictions saved to 'nokia_predicted_output.csv'")


Device set to use cpu


✅ Predictions saved to 'nokia_predicted_output.csv'


In [18]:
df['predictedemotion'].value_counts()

predictedemotion
disgust     1421
neutral     1282
anger       1262
sadness      194
fear         140
surprise     116
joy            6
Name: count, dtype: int64

In [19]:
model_dir = "./emotion-model"

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Create pipeline for prediction
emotion_classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

# --------------------------
# 2. Load Unseen Text Data
# --------------------------
# Replace with your file path
df = pd.read_csv("canadaaspectsentiment.csv")  # Should have a column named "text"

# Drop rows with missing text
df = df.dropna(subset=['body'])

# --------------------------
# 3. Predict Emotions
# --------------------------
predictions = emotion_classifier(df['body'].tolist(), truncation=True)

# Add predicted labels to dataframe
df['predictedemotion'] = [pred['label'] for pred in predictions]

# --------------------------
# 4. Save Results
# --------------------------
df.to_csv("canada_predicted_output.csv", index=False)
print("✅ Predictions saved to 'canada_predicted_output.csv'")

Device set to use cpu


✅ Predictions saved to 'canada_predicted_output.csv'


In [20]:
df['predictedemotion'].value_counts()

predictedemotion
anger       828
neutral     740
disgust     182
surprise    129
sadness      69
fear         19
joy          10
Name: count, dtype: int64

In [21]:
model_dir = "./emotion-model"

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Create pipeline for prediction
emotion_classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

# --------------------------
# 2. Load Unseen Text Data
# --------------------------
# Replace with your file path
df = pd.read_csv("flintaspectfinetunefinal.csv")  # Should have a column named "text"

# Drop rows with missing text
df = df.dropna(subset=['body'])

# --------------------------
# 3. Predict Emotions
# --------------------------
predictions = emotion_classifier(df['body'].tolist(), truncation=True)

# Add predicted labels to dataframe
df['predictedemotion'] = [pred['label'] for pred in predictions]

# --------------------------
# 4. Save Results
# --------------------------
df.to_csv("usa_predicted_output.csv", index=False)
print("✅ Predictions saved to 'usa_predicted_output.csv'")

Device set to use cpu


✅ Predictions saved to 'usa_predicted_output.csv'


In [22]:
df['predictedemotion'].value_counts()

predictedemotion
anger       14615
neutral      4933
disgust      1402
surprise     1049
sadness       540
fear          140
joy            45
Name: count, dtype: int64

In [27]:
model_dir = "./emotion-model"

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Create pipeline for prediction
emotion_classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)


# --------------------------
# 2. Load Unseen Text Data
# --------------------------
# Replace with your file path
df = pd.read_csv("predictedemotionalleval.csv")  # Should have a column named "text"

# Drop rows with missing text
df = df.dropna(subset=['body'])

# --------------------------
# 3. Predict Emotions
# --------------------------
predictions = emotion_classifier(df['body'].tolist(), truncation=True)

# Add predicted labels to dataframe
df['predictedemotionft'] = [pred['label'] for pred in predictions]

# --------------------------
# 4. Save Results
# --------------------------
df.to_csv("eval_data.csv", index=False)
print("Predictions saved to eval_data.csv") 



Device set to use cpu


Predictions saved to eval_data.csv


In [28]:
df.columns

Index(['Unnamed: 0.14', 'Unnamed: 0.13', 'Unnamed: 0.12', 'Unnamed: 0.11',
       'Unnamed: 0.10', 'Unnamed: 0.9', 'Unnamed: 0.8', 'Unnamed: 0.7',
       'Unnamed: 0.6', 'Unnamed: 0.5', 'Unnamed: 0.4', 'Unnamed: 0.3',
       'Unnamed: 0.2', 'Unnamed: 0', 'Unnamed: 0.1', 'msg_type', 'datetime',
       'title', 'thread_id', 'comment_id', 'topic_name_top', 'topic_name_leaf',
       'thread_text', 'date', 'body', 'emotion', 'llm_emotion', 'normllm_emo',
       'government_sentiment', 'health_sentiment', 'community_sentiment',
       'misinformation_sentiment', 'overall_sentiment', 'title_translated',
       'overall_sentiment_label', 'government_sentiment_label',
       'health_sentiment_label', 'community_sentiment_label',
       'misinformation_sentiment_label', 'processed_thread_text',
       'transprocessed_thread_text', 'fin_sentiment_score',
       'eng_sentiment_score', 'fin_sentiment_label', 'eng_sentiment_label',
       'topic_name', 'llmtopic_name', 'fisentiment', 'ensentiment',


In [31]:
df.rename(columns={'predicted_emotion': 'predicted_emotion_llm', 'emotion': 'emotionwithoutft'}, inplace=True)


In [36]:
df = df[df['predicted_emotion_llm'] != 'concern']


In [44]:
from transformers import pipeline
import pandas as pd
from tqdm import tqdm

# Initialize emotion classifier with truncation enabled
classifier = pipeline(
    "sentiment-analysis",
    model="michellejieli/emotion_text_classifier",
    truncation=True,        # truncate long texts safely
    padding=True,           # pad shorter texts
    max_length=512,         # enforce BERT max token limit
    batch_size=16           # process in small batches for efficiency
)

# Example: assume df is already loaded and has a 'body' column
# df = pd.read_csv("your_file.csv")

# Apply the classifier
tqdm.pandas(desc="Classifying emotions")

def get_emotion(text):
    if pd.isnull(text) or not str(text).strip():
        return None
    try:
        result = classifier(str(text))[0]
        return result["label"]
    except Exception as e:
        print(f"Error on text: {text[:60]}... -> {e}")
        return None

df["nofinetune_emotion2"] = df["body"].progress_apply(get_emotion)

# View results
print(df.head())



Device set to use cpu
Classifying emotions: 100%|████████████████████████████████████████████████████████| 2979/2979 [00:57<00:00, 52.05it/s]

   Unnamed: 0.14  Unnamed: 0.13  Unnamed: 0.12  Unnamed: 0.11  Unnamed: 0.10  \
0              0            NaN            NaN            NaN            NaN   
1              1            NaN            NaN            NaN            NaN   
2              2            NaN            NaN            NaN            NaN   
3              3            NaN            NaN            NaN            NaN   
4              4            NaN            NaN            NaN            NaN   

   Unnamed: 0.9  Unnamed: 0.8  Unnamed: 0.7  Unnamed: 0.6  Unnamed: 0.5  ...  \
0           NaN           NaN           NaN           NaN           NaN  ...   
1           NaN           NaN           NaN           NaN           NaN  ...   
2           NaN           NaN           NaN           NaN           NaN  ...   
3           NaN           NaN           NaN           NaN           NaN  ...   
4           NaN           NaN           NaN           NaN           NaN  ...   

   score  url          created_utc  nu

In [43]:
df["nofinetune_emotion"].value_counts()

nofinetune_emotion
neutral     2153
disgust      279
anger        249
surprise     118
joy           91
sadness       67
fear          22
Name: count, dtype: int64

In [45]:
df["nofinetune_emotion2"].value_counts()

nofinetune_emotion2
neutral     2153
disgust      279
anger        249
surprise     118
joy           91
sadness       67
fear          22
Name: count, dtype: int64

In [37]:
#Before Fine-Tuning

In [40]:
df['emotionwithoutft'].value_counts()

emotionwithoutft
neutral     392
disgust      34
anger        19
joy          18
surprise     10
fear          5
sadness       2
Name: count, dtype: int64

In [38]:
df['predicted_emotion_llm'].value_counts()

predicted_emotion_llm
anger       1552
neutral      785
disgust      224
surprise     198
sadness      143
fear          67
joy           10
Name: count, dtype: int64

In [46]:
import pandas as pd
from sklearn.metrics import classification_report

# Load your dataframes
#df_pred = pd.read_csv("/content/predicted_output.csv")       # Contains predictedemotion
#df_true = pd.read_csv("/content/filteredemotion_dataset.csv")           # Contains predicted_emotion

# Merge on thread_text_translated
#merged_df = pd.merge(df_true, df_pred, on="thread_text_translated", how="inner")

# Get ground truth and predicted labels
y_true = df["predicted_emotion_llm"]
y_pred = df["nofinetune_emotion"]

# Print class-wise metrics
report = classification_report(y_true, y_pred, digits=3)
print(report)


              precision    recall  f1-score   support

       anger      0.932     0.149     0.258      1552
     disgust      0.140     0.174     0.155       224
        fear      0.409     0.134     0.202        67
         joy      0.099     0.900     0.178        10
     neutral      0.338     0.926     0.495       785
     sadness      0.373     0.175     0.238       143
    surprise      0.237     0.141     0.177       198

    accuracy                          0.359      2979
   macro avg      0.361     0.371     0.243      2979
weighted avg      0.628     0.359     0.305      2979



In [47]:
#After Fine-Tuning

In [48]:
y_true = df["predicted_emotion_llm"]
y_pred = df["predictedemotionft"]

# Print class-wise metrics
report = classification_report(y_true, y_pred, digits=3)
print(report)

              precision    recall  f1-score   support

       anger      0.765     0.903     0.828      1552
     disgust      0.416     0.531     0.467       224
        fear      0.387     0.179     0.245        67
         joy      0.444     0.400     0.421        10
     neutral      0.710     0.555     0.623       785
     sadness      0.378     0.217     0.276       143
    surprise      0.452     0.288     0.352       198

    accuracy                          0.692      2979
   macro avg      0.508     0.439     0.459      2979
weighted avg      0.675     0.692     0.674      2979



In [ ]:
df_true.columns

Index(['Unnamed: 0', 'thread_text_translated', 'predicted_aspect',
       'predicted_sentiment', 'predicted_emotion'],
      dtype='object')

In [ ]:
df = pd.read_csv('/content/unlabeled_data_with_predictions.csv')

In [ ]:
df.columns

Index(['text', 'predicted_labels'], dtype='object')

In [ ]:
df.head()

,text,predicted_labels
0,Food all.ies have not been studied. I’ve been ...,"food:negative, symptoms:negative, multi-vitami..."
1,"From the very beginning, it has been a bit of ...","heat burning:negative, water:neutral, food:neu..."
2,Sounds terrible. Hopefully there are no more m...,"Botox:negative, pain treatment:negative, water..."
3,"Moi! Luckily, these “kaic treatments” ended af...","botulin treatments:negative, botulin:negative,..."
4,"The zinced tank is its own ""water wheelbase"" p...","water wheelbase:neutral, heating:neutral, pump..."


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df)

https://docs.google.com/spreadsheets/d/1Z0gemb0WoNP2C1nf4us108wPH7vJgju-xOmurWchEk0/edit#gid=0


In [ ]:
import pandas as pd
df = pd.read_csv('/content/predicted_output.csv')
df.columns

Index(['Unnamed: 0.12', 'Unnamed: 0.11', 'Unnamed: 0.10', 'Unnamed: 0.9',
       'Unnamed: 0.8', 'Unnamed: 0.7', 'Unnamed: 0.6', 'Unnamed: 0.5',
       'Unnamed: 0.4', 'Unnamed: 0.3', 'Unnamed: 0.2', 'Unnamed: 0',
       'Unnamed: 0.1', 'msg_type', 'datetime', 'title', 'thread_id',
       'comment_id', 'topic_name_top', 'topic_name_leaf', 'thread_text',
       'date', 'thread_text_translated', 'emotion', 'llm_emotion',
       'normllm_emo', 'government_sentiment', 'health_sentiment',
       'community_sentiment', 'misinformation_sentiment', 'overall_sentiment',
       'title_translated', 'overall_sentiment_label',
       'government_sentiment_label', 'health_sentiment_label',
       'community_sentiment_label', 'misinformation_sentiment_label',
       'processed_thread_text', 'transprocessed_thread_text',
       'fin_sentiment_score', 'eng_sentiment_score', 'fin_sentiment_label',
       'eng_sentiment_label', 'topic_name', 'llmtopic_name', 'fisentiment',
       'ensentiment', 'bleu_sco

In [ ]:
df['emotion'].value_counts()

,count
emotion,
neutral,3698
disgust,283
joy,140
anger,133
surprise,101
sadness,34
fear,32
